[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module3/02-web-scraping.ipynb)

# Module 3.2 — Web Scraping
**Module 3: Automation & Scripting** | Estimated time: 30 minutes

---

## Learning Objectives
By the end of this notebook you will be able to:
- Fetch web pages with the `requests` library including headers and timeouts
- Parse HTML with `BeautifulSoup` using `find`, `find_all`, and CSS selectors
- Extract text, attributes, and links from a real web page
- Handle multi-page scraping with pagination logic
- Check `robots.txt` before scraping with `urllib.robotparser`
- Apply rate limiting with `time.sleep` to be a polite scraper
- Save scraped data to a JSON file

In [ ]:
!pip install requests beautifulsoup4 -q

import requests
from bs4 import BeautifulSoup
import json
import time
import urllib.robotparser
from pathlib import Path

print('Libraries ready.')

## 1. Making HTTP Requests with `requests`

Always pass a `User-Agent` header so the server knows you are a script, and set a `timeout` so your program does not hang forever.

In [ ]:
HEADERS = {
    'User-Agent': 'PyPath-Scraper/1.0 (educational project; +https://github.com/your-org/pypath)'
}

url = 'https://books.toscrape.com/'
response = requests.get(url, headers=HEADERS, timeout=10)

# Status code guide:
# 200 OK  |  301/302 Redirect  |  403 Forbidden  |  404 Not Found  |  429 Rate Limited  |  500 Server Error
print('Status code :', response.status_code)
print('Content-Type:', response.headers.get('Content-Type', 'N/A'))
print('Page size   :', len(response.text), 'characters')
print('Encoding    :', response.encoding)
print()

# Always raise for bad statuses instead of silently continuing
try:
    response.raise_for_status()
    print('Request succeeded.')
except requests.HTTPError as e:
    print('HTTP error:', e)

## 2. Parsing HTML with BeautifulSoup

BeautifulSoup turns raw HTML into a navigable tree. Key methods:

| Method | Returns | Use when |
|---|---|---|
| `soup.find(tag, attrs)` | First match or `None` | You want one element |
| `soup.find_all(tag, attrs)` | List of all matches | You want every element |
| `soup.select(css_selector)` | List of matches | You prefer CSS syntax |
| `soup.select_one(css_selector)` | First match or `None` | CSS, single result |

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')

# Page title
title = soup.find('title')
print('Page title:', title.get_text(strip=True))
print()

# Find total number of books
total_el = soup.select_one('form strong')
if total_el:
    print('Total books in catalogue:', total_el.get_text(strip=True))
print()

# All book article elements on this page
books = soup.find_all('article', class_='product_pod')
print(f'Books on page 1: {len(books)}')
print()

# Inspect first book
first = books[0]
print('--- First book ---')
print('Title  :', first.h3.a['title'])
print('Price  :', first.select_one('.price_color').get_text(strip=True))
print('Rating :', first.p['class'][1])  # e.g. ['star-rating', 'Three']
print('In stock:', first.select_one('.availability').get_text(strip=True))

## 3. Extracting Data from All Books on the Page

In [ ]:
RATING_MAP = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

def parse_books(soup_obj: BeautifulSoup, base_url: str) -> list[dict]:
    """Extract book data from one catalogue page."""
    results = []
    for article in soup_obj.find_all('article', class_='product_pod'):
        title   = article.h3.a['title']
        price   = article.select_one('.price_color').get_text(strip=True)
        rating_word = article.p['class'][1]  # 'One'..'Five'
        rating  = RATING_MAP.get(rating_word, 0)
        in_stock = 'in stock' in article.select_one('.availability').get_text().lower()
        href    = article.h3.a['href'].replace('../', '')
        link    = base_url + 'catalogue/' + href.split('catalogue/')[-1]
        results.append({
            'title'   : title,
            'price'   : price,
            'rating'  : rating,
            'in_stock': in_stock,
            'url'     : link,
        })
    return results

page1_books = parse_books(soup, 'https://books.toscrape.com/')
print(f'Parsed {len(page1_books)} books from page 1.')
for b in page1_books[:3]:
    print(f"  {b['rating']}★  {b['price']}  {b['title'][:50]}")

## 4. Handling Pagination

Most catalogue sites use a "next" button. We follow the link until there is no more "next" page.

In [ ]:
def get_next_page_url(soup_obj: BeautifulSoup, current_url: str) -> str | None:
    """Return the URL of the next page, or None if we are on the last page."""
    next_btn = soup_obj.select_one('li.next a')
    if next_btn is None:
        return None
    href = next_btn['href']
    # href is relative, e.g. 'catalogue/page-2.html'
    base = 'https://books.toscrape.com/'
    if href.startswith('catalogue/'):
        return base + href
    return base + 'catalogue/' + href


# Scrape first 3 pages only (polite demo — full catalogue has 50 pages)
MAX_PAGES = 3
RATE_LIMIT_SECONDS = 1.0  # wait between requests

all_books = []
current_url = 'https://books.toscrape.com/'
page_num = 0

while current_url and page_num < MAX_PAGES:
    page_num += 1
    print(f'Scraping page {page_num}: {current_url}')
    resp = requests.get(current_url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    page_soup = BeautifulSoup(resp.text, 'html.parser')
    books_on_page = parse_books(page_soup, 'https://books.toscrape.com/')
    all_books.extend(books_on_page)
    current_url = get_next_page_url(page_soup, current_url)
    if current_url:
        time.sleep(RATE_LIMIT_SECONDS)  # be polite!

print(f'\nTotal books scraped: {len(all_books)}')

## 5. Checking `robots.txt` Before Scraping

Always verify that the site allows scraping. `urllib.robotparser` parses the `robots.txt` file automatically.

In [ ]:
rp = urllib.robotparser.RobotFileParser()
rp.set_url('https://books.toscrape.com/robots.txt')
rp.read()

test_urls = [
    'https://books.toscrape.com/',
    'https://books.toscrape.com/catalogue/page-2.html',
    'https://books.toscrape.com/admin/',
]
UA = 'PyPath-Scraper'
for u in test_urls:
    allowed = rp.can_fetch(UA, u)
    crawl_delay = rp.crawl_delay(UA)
    print(f'  {"ALLOW" if allowed else "DENY ":6s}  {u}')
    if crawl_delay:
        print(f'         Crawl-delay for {UA}: {crawl_delay}s')

## 6. Extracting Links from a Page

In [ ]:
# Collect all unique hrefs from the catalogue page
all_links = set()
for tag in soup.find_all('a', href=True):
    href = tag['href']
    if href.startswith('http'):
        all_links.add(href)
    elif href.startswith('catalogue/'):
        all_links.add('https://books.toscrape.com/' + href)

print(f'Unique links found on page 1: {len(all_links)}')
# Show first 5
for link in sorted(all_links)[:5]:
    print(' ', link)

## 7. Saving Scraped Data to JSON

In [ ]:
output_path = Path('/tmp/scraped_books.json')

with output_path.open('w', encoding='utf-8') as f:
    json.dump({'total': len(all_books), 'books': all_books}, f, indent=2, ensure_ascii=False)

print(f'Saved {len(all_books)} books to {output_path}')
print(f'File size: {output_path.stat().st_size:,} bytes')
print()

# Read back and display stats
with output_path.open() as f:
    data = json.load(f)

ratings = [b['rating'] for b in data['books']]
avg_rating = sum(ratings) / len(ratings)
prices = [float(b['price'].replace('\u00a3', '')) for b in data['books']]
avg_price = sum(prices) / len(prices)

print('Summary:')
print(f'  Average rating : {avg_rating:.2f} / 5')
print(f'  Average price  : \u00a3{avg_price:.2f}')
print(f'  In stock       : {sum(1 for b in data["books"] if b["in_stock"])} / {len(data["books"])}')

## Practice Exercises

**Exercise 1 — Category Scraper**  
The books.toscrape.com sidebar lists categories (Fiction, Mystery, etc.). Write a function `get_categories(soup) -> dict[str, str]` that returns a mapping of `{category_name: url}` for every category listed in the sidebar `<ul>` under the `<div class="side_categories">` element.

**Exercise 2 — Highest Rated Books**  
Using the `all_books` list you already scraped, print the top 5 highest-rated books. If there are ties in rating, sort alphabetically by title.

**Exercise 3 — Polite Scraper Class**  
Create a class `PoliteScraper` with:
- `__init__(self, base_url, delay=1.0)` — stores the base URL and rate limit
- `can_scrape(url) -> bool` — checks robots.txt (cache the result)
- `get(url) -> BeautifulSoup | None` — fetches the URL only if allowed, waits `delay` seconds between requests, returns parsed soup or `None`